In [1]:
import optuna
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import lightgbm as lgb

from datetime import datetime
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error



/Users/sziadat/miniforge3/envs/bbbx_env_revisions/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SEED = 15

# Set dataset (model) [options: full_dataset, maccs, mordred, ecfp, ecfp_mordred, ecfp_maccs, mordred_maccs]
ds = 'full_dataset'
full_dataset_path = '../datasets/druglike_b3db_labelled.csv'

In [5]:
# Import Dataset
full_df = pd.read_csv(full_dataset_path)

smiles = full_df.loc[:, 'SMILES']
bbb_class = full_df.loc[:, 'Class']
logbb = full_df.loc[:, 'logBB']

starting_col = list(full_df.columns).index('Class') + 1
X_full = full_df.iloc[:, starting_col:]

bbb_class = np.array(bbb_class)
logbb = np.array(logbb)


if ds == 'full_dataset': 
    X_full = X_full
    col_names = list(full_df.columns)[starting_col:]
elif ds == 'ecfp': 
    X_full = X_full.iloc[:, :2048]
    col_names = list(full_df.columns)[starting_col:][:2048]
elif ds == 'maccs': 
    X_full = X_full.iloc[:, -167:]
    col_names = list(full_df.columns)[starting_col:][-167:]
elif ds == 'mordred': 
    X_full = X_full.iloc[:, 2048:-167]
    col_names = list(full_df.columns)[starting_col:][2048:-167]
elif ds == 'ecfp_mordred': 
    X_full = X_full.iloc[:, :-167]
    col_names = list(full_df.columns)[starting_col:][:-167]
elif ds == 'mordred_maccs': 
    X_full = X_full.iloc[:, 2048:]
    col_names = list(full_df.columns)[starting_col:][2048:]
elif ds == 'ecfp_maccs': 
    X_full = X_full.drop(X_full.columns[2048:-167], axis=1)
    col_names = list(full_df.columns)[starting_col:][:2048] + list(full_df.columns)[starting_col:][-167:]
else: 
    print('Invalid dataset specified, proceeding with the full dataset')


X_full = X_full.values
X_to_drop = pd.DataFrame(data=X_full, columns=col_names)
X_full_no_nan = X_to_drop.dropna(axis=1)
X_full_no_nan = X_full_no_nan.loc[:, X_full_no_nan.std() != 0]

X_all = X_full_no_nan.values

In [6]:
import numpy as np
import pandas as pd

mask = (
    np.isfinite(logbb)
    & np.isfinite(bbb_class)
)

threshold_class = (
    logbb[mask] < -1.0
).astype(int)

observed_class = (
    bbb_class[mask]
    .astype(int)
)

print(
    pd.crosstab(
        observed_class,
        threshold_class,
        rownames=["Recorded BBB class"],
        colnames=["Class implied by logBB"],
    )
)

agreement = np.mean(
    observed_class == threshold_class
)

print(f"Agreement: {agreement:.3f}")
print(f"Disagreement: {1 - agreement:.3f}")

Class implied by logBB    0   1
Recorded BBB class             
0                       678   0
1                         0  87
Agreement: 1.000
Disagreement: 0.000


In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold


# ============================================================
# Configuration
# ============================================================

n_outer = 5
max_logbb_bins = 5

outerfold_csv_path = Path(
    "../datasets/druglike_b3db_outerfolds.csv"
)


# ============================================================
# Generate distribution-balanced outer-fold assignments
# ============================================================

def generate_distribution_balanced_outer_folds(
    bbb_class,
    logbb,
    n_splits=5,
    max_bins=5,
    random_state=42,
):
    """
    Generate outer folds that balance:

    1. the number of compounds with measured logBB;
    2. the distribution of measured logBB values;
    3. BBB class among labeled compounds, where possible;
    4. BBB class among compounds without measured logBB.

    The function tries increasingly less granular stratification
    schemes until every stratum contains at least n_splits samples.

    Returns
    -------
    outer_fold_assignment : np.ndarray
        Fold number assigned to each original row.

    outer_splits : list[tuple[np.ndarray, np.ndarray]]
        List of (train_idx, test_idx) pairs.

    outer_strata : np.ndarray
        Final stratification labels used.

    stratification_description : str
        Description of the selected stratification scheme.
    """

    bbb_class = np.asarray(bbb_class)
    logbb = np.asarray(logbb, dtype=float)

    if len(bbb_class) != len(logbb):
        raise ValueError(
            "bbb_class and logbb must have the same number of rows."
        )

    n_samples = len(logbb)
    has_logbb = np.isfinite(logbb)

    if has_logbb.sum() < n_splits:
        raise ValueError(
            f"Only {has_logbb.sum()} compounds have measured logBB, "
            f"which is fewer than n_splits={n_splits}."
        )

    if pd.isna(bbb_class).any():
        raise ValueError(
            "bbb_class contains missing values. All rows must have "
            "a classification label for stratified splitting."
        )

    # Convert class labels to strings so this also works if the
    # labels are not strictly integers.
    class_strings = pd.Series(bbb_class).astype(str).to_numpy()

    labeled_indices = np.flatnonzero(has_logbb)
    unlabeled_indices = np.flatnonzero(~has_logbb)

    candidate_schemes = []

    # --------------------------------------------------------
    # Scheme 1:
    # Labeled: BBB class × logBB quantile
    # Unlabeled: BBB class
    # --------------------------------------------------------

    for requested_bins in range(max_bins, 1, -1):

        labeled_bins = pd.qcut(
            logbb[has_logbb],
            q=requested_bins,
            labels=False,
            duplicates="drop",
        )

        labeled_bins = np.asarray(labeled_bins, dtype=int)
        actual_bins = len(np.unique(labeled_bins))

        strata = np.empty(n_samples, dtype=object)

        for row_index, class_label, bin_label in zip(
            labeled_indices,
            class_strings[has_logbb],
            labeled_bins,
        ):
            strata[row_index] = (
                f"labeled_class_{class_label}"
                f"_logbb_bin_{bin_label}"
            )

        for row_index, class_label in zip(
            unlabeled_indices,
            class_strings[~has_logbb],
        ):
            strata[row_index] = (
                f"unlabeled_class_{class_label}"
            )

        candidate_schemes.append(
            (
                strata,
                (
                    "measured compounds stratified by BBB class "
                    f"and {actual_bins} logBB quantile bins; "
                    "unmeasured compounds stratified by BBB class"
                ),
            )
        )

    # --------------------------------------------------------
    # Scheme 2:
    # Labeled: logBB quantile only
    # Unlabeled: BBB class
    #
    # This is useful if class × quantile strata are too small.
    # --------------------------------------------------------

    for requested_bins in range(max_bins, 1, -1):

        labeled_bins = pd.qcut(
            logbb[has_logbb],
            q=requested_bins,
            labels=False,
            duplicates="drop",
        )

        labeled_bins = np.asarray(labeled_bins, dtype=int)
        actual_bins = len(np.unique(labeled_bins))

        strata = np.empty(n_samples, dtype=object)

        for row_index, bin_label in zip(
            labeled_indices,
            labeled_bins,
        ):
            strata[row_index] = (
                f"labeled_logbb_bin_{bin_label}"
            )

        for row_index, class_label in zip(
            unlabeled_indices,
            class_strings[~has_logbb],
        ):
            strata[row_index] = (
                f"unlabeled_class_{class_label}"
            )

        candidate_schemes.append(
            (
                strata,
                (
                    "measured compounds stratified by "
                    f"{actual_bins} logBB quantile bins; "
                    "unmeasured compounds stratified by BBB class"
                ),
            )
        )

    # --------------------------------------------------------
    # Scheme 3:
    # BBB class × measured-logBB availability
    #
    # This balances labeled counts and classes but not the
    # detailed continuous logBB distribution.
    # --------------------------------------------------------

    availability_class_strata = np.array(
        [
            (
                f"class_{class_label}_"
                f"has_logbb_{int(is_labeled)}"
            )
            for class_label, is_labeled in zip(
                class_strings,
                has_logbb,
            )
        ],
        dtype=object,
    )

    candidate_schemes.append(
        (
            availability_class_strata,
            (
                "BBB class and measured-logBB availability; "
                "continuous logBB distribution could not be "
                "stratified because some strata were too small"
            ),
        )
    )

    # --------------------------------------------------------
    # Scheme 4:
    # Measured-logBB availability only.
    #
    # Last-resort fallback that guarantees nearly equal numbers
    # of labeled compounds per outer fold.
    # --------------------------------------------------------

    availability_only_strata = np.where(
        has_logbb,
        "has_logbb",
        "no_logbb",
    )

    candidate_schemes.append(
        (
            availability_only_strata,
            (
                "measured-logBB availability only; more detailed "
                "stratification was not possible"
            ),
        )
    )

    selected_strata = None
    selected_description = None

    for candidate_strata, description in candidate_schemes:

        stratum_counts = (
            pd.Series(candidate_strata)
            .value_counts()
        )

        if stratum_counts.min() >= n_splits:
            selected_strata = candidate_strata
            selected_description = description
            break

    if selected_strata is None:
        raise ValueError(
            "Could not construct valid stratification labels. "
            "At least one required group contains fewer samples "
            f"than n_splits={n_splits}."
        )

    print("Selected outer-fold stratification:")
    print(selected_description)

    print("\nStratum counts:")
    print(
        pd.Series(selected_strata)
        .value_counts()
        .sort_index()
    )

    outer_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    outer_fold_assignment = np.full(
        n_samples,
        fill_value=-1,
        dtype=int,
    )

    # X is not used by StratifiedKFold, so a placeholder array
    # is sufficient.
    placeholder_X = np.zeros((n_samples, 1))

    for outer_fold, (_, test_idx) in enumerate(
        outer_cv.split(
            placeholder_X,
            selected_strata,
        )
    ):
        outer_fold_assignment[test_idx] = outer_fold

    if np.any(outer_fold_assignment < 0):
        missing_rows = np.flatnonzero(
            outer_fold_assignment < 0
        )

        raise RuntimeError(
            "Some rows were not assigned to an outer fold: "
            f"{missing_rows[:20]}"
        )

    # Reconstruct explicit train/test index lists from the saved
    # assignment. These are what the training loop should use.
    outer_splits = []

    for outer_fold in range(n_splits):

        test_idx = np.flatnonzero(
            outer_fold_assignment == outer_fold
        )

        train_idx = np.flatnonzero(
            outer_fold_assignment != outer_fold
        )

        outer_splits.append(
            (train_idx, test_idx)
        )

    return (
        outer_fold_assignment,
        outer_splits,
        selected_strata,
        selected_description,
    )


(
    outer_fold_assignment,
    outer_splits,
    outer_strata,
    outer_stratification_description,
) = generate_distribution_balanced_outer_folds(
    bbb_class=bbb_class,
    logbb=logbb,
    n_splits=n_outer,
    max_bins=max_logbb_bins,
    random_state=SEED,
)


# ============================================================
# Validate the generated outer folds
# ============================================================

has_logbb = np.isfinite(logbb)

fold_validation_df = pd.DataFrame(
    {
        "outer_fold": outer_fold_assignment,
        "bbb_class": bbb_class,
        "has_logbb": has_logbb,
        "logBB": logbb,
        "outer_stratum": outer_strata,
    }
)

print("\nSamples per outer fold:")
print(
    fold_validation_df
    .groupby("outer_fold")
    .size()
)

print("\nMeasured logBB samples per outer fold:")
print(
    fold_validation_df
    .groupby("outer_fold")["has_logbb"]
    .sum()
)

print("\nLabeled and unlabeled counts per fold:")
print(
    pd.crosstab(
        fold_validation_df["outer_fold"],
        fold_validation_df["has_logbb"],
    ).rename(
        columns={
            False: "without_logBB",
            True: "with_logBB",
        }
    )
)

print("\nBBB-class counts per outer fold:")
print(
    pd.crosstab(
        fold_validation_df["outer_fold"],
        fold_validation_df["bbb_class"],
    )
)

print("\nBBB class × logBB availability per fold:")
print(
    fold_validation_df.groupby(
        [
            "outer_fold",
            "bbb_class",
            "has_logbb",
        ]
    )
    .size()
    .unstack(
        ["bbb_class", "has_logbb"],
        fill_value=0,
    )
)

print("\nMeasured logBB distribution per outer fold:")
print(
    fold_validation_df.loc[
        fold_validation_df["has_logbb"]
    ]
    .groupby("outer_fold")["logBB"]
    .agg(
        [
            "count",
            "mean",
            "std",
            "min",
            "median",
            "max",
        ]
    )
)


# ============================================================
# Save the outer-fold assignments with the full dataset
# ============================================================

n_samples = len(logbb)

if len(smiles) != n_samples:
    raise ValueError(
        "smiles and logbb do not have the same number of rows."
    )

if len(X_full_no_nan) != n_samples:
    raise ValueError(
        "X_full_no_nan and logbb do not have the same number "
        "of rows."
    )

# Preserve descriptor names when X_full_no_nan is already a
# pandas DataFrame.
if isinstance(X_full_no_nan, pd.DataFrame):

    descriptor_df = (
        X_full_no_nan
        .reset_index(drop=True)
        .copy()
    )

else:

    X_full_no_nan_array = np.asarray(
        X_full_no_nan
    )

    descriptor_df = pd.DataFrame(
        X_full_no_nan_array,
        columns=[
            f"descriptor_{feature_index}"
            for feature_index in range(
                X_full_no_nan_array.shape[1]
            )
        ],
    )

# Convert descriptor failures or nonnumeric values to NaN.
descriptor_df = descriptor_df.apply(
    pd.to_numeric,
    errors="coerce",
)

metadata_df = pd.DataFrame(
    {
        "sample_id": np.arange(n_samples),
        "smiles": np.asarray(smiles),
        "logBB": np.asarray(logbb, dtype=float),
        "bbb_class": np.asarray(bbb_class),
        "has_logbb": has_logbb,
        "outer_fold": outer_fold_assignment,
        "outer_stratum": outer_strata,
    }
)

reserved_columns = set(
    metadata_df.columns
)

conflicting_descriptor_columns = (
    reserved_columns.intersection(
        descriptor_df.columns
    )
)

if conflicting_descriptor_columns:
    raise ValueError(
        "The following descriptor column names conflict with "
        "metadata columns: "
        f"{sorted(conflicting_descriptor_columns)}"
    )

dataset_with_outer_folds = pd.concat(
    [
        metadata_df,
        descriptor_df,
    ],
    axis=1,
)

outerfold_csv_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

dataset_with_outer_folds.to_csv(
    outerfold_csv_path,
    index=False,
)

print(
    "\nSaved distribution-balanced outer folds to:"
)

print(
    outerfold_csv_path.resolve()
)


# ============================================================
# Use these folds in the training loop
# ============================================================

# Replace:
#
# for train_idx, test_idx in outer_cv.split(
#     X_all,
#     bbb_class,
# ):
#
# with:

for count_outer, (train_idx, test_idx) in enumerate(
    outer_splits,
    start=1,
):

    X_train = X_all[train_idx]
    X_test = X_all[test_idx]

    y_train_c = bbb_class[train_idx]
    y_test_c = bbb_class[test_idx]

    y_train_r = logbb[train_idx]
    y_test_r = logbb[test_idx]

    smiles_train = smiles[train_idx]
    smiles_test = smiles[test_idx]

    print(
        f"\nOuter fold {count_outer}/{n_outer}"
    )

    print(
        f"Training samples: {len(train_idx)}"
    )

    print(
        f"Test samples: {len(test_idx)}"
    )

    print(
        "Training samples with measured logBB: "
        f"{np.isfinite(y_train_r).sum()}"
    )

    print(
        "Test samples with measured logBB: "
        f"{np.isfinite(y_test_r).sum()}"
    )

    # Continue with your existing inner-CV and model-training
    # code here.

Selected outer-fold stratification:
measured compounds stratified by BBB class and 5 logBB quantile bins; unmeasured compounds stratified by BBB class

Stratum counts:
labeled_class_0_logbb_bin_0      69
labeled_class_0_logbb_bin_1     151
labeled_class_0_logbb_bin_2     155
labeled_class_0_logbb_bin_3     153
labeled_class_0_logbb_bin_4     150
labeled_class_1_logbb_bin_0      87
unlabeled_class_0              1517
unlabeled_class_1               874
Name: count, dtype: int64

Samples per outer fold:
outer_fold
0    632
1    631
2    631
3    631
4    631
dtype: int64

Measured logBB samples per outer fold:
outer_fold
0    153
1    153
2    153
3    153
4    153
Name: has_logbb, dtype: int64

Labeled and unlabeled counts per fold:
has_logbb   without_logBB  with_logBB
outer_fold                           
0                     479         153
1                     478         153
2                     478         153
3                     478         153
4                     478     